# **Hands-on Practice: ABSA (Aspect-Based Sentiment Analysis)**

## **1. Load Dataset**

### (1) import library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import torch
from google.colab import files

from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder

### (2) setting device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### (3) load data

* Laptop_Review_with_aspect.csv

In [ ]:
uploaded = files.upload()
data_path = list(uploaded.keys())[0]

In [ ]:
data = pd.read_csv(data_path)
data.head()

In [ ]:
sns.countplot(x='aspect', data = data, hue = 'aspect')
plt.grid()
plt.xticks(rotation=90)
plt.show()

In [ ]:
sns.countplot(x='aspect', data = data, hue = 'label')
plt.grid()
plt.xticks(rotation=90)
plt.show()

## **2. Pre-processing Data**

### (1) Integer encoding : y

In [ ]:
le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])
data.head()

In [ ]:
label_list = le.classes_
label_list

### (2) Pre-processing Input Data
* [ASPECT]aspect[SEP]review

In [ ]:
# data split
train, val = train_test_split(data, test_size=0.2, random_state=100)
val_org = val.copy()  # for aspect-level evaluation

# Combine aspect and text into a single input format
train['text'] = '[ASPECT]' + train['aspect'] + '[SEP]' + train['text']
val['text'] = '[ASPECT]' + val['aspect'] + '[SEP]' + val['text']

# Keep only the text and label columns
train = train.loc[:, ['text', 'label']]
val = val.loc[:, ['text', 'label']]

train.head()

### (3) Dataset

In [ ]:
train_ts = Dataset.from_pandas(train)
val_ts = Dataset.from_pandas(val)

## **3. Tokenization**

### (1) download tokenizer

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

### (2) add special token

In [ ]:
# Define custom special tokens to mark the aspect in the input text
spec_tokens = {"additional_special_tokens": ["[ASPECT]"]}

# Add the special tokens to the tokenizer's vocabulary
num_added_toks = tokenizer.add_special_tokens(spec_tokens)

print(f"Added {num_added_toks} special tokens: {spec_tokens['additional_special_tokens']}")

### (3) tokenization

In [ ]:
def preprocess_function(data):
    # Tokenize the input text with fixed max length
    tokenized = tokenizer(data['text'], padding="max_length", truncation=True, max_length=128)
    # Attach the label to the tokenized output
    tokenized["label"] = int(data["label"])
    return tokenized

# Apply preprocessing to each example in the train and validation sets
train_ts = train_ts.map(preprocess_function, batched=False)
val_ts = val_ts.map(preprocess_function, batched=False)

## **4. Load Pre-trained Model**

In [ ]:
# Load the pre-trained model with 2 sentiment labels
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# Resize embeddings to include the added special token
model.resize_token_embeddings(len(tokenizer))

## **5. Set Training Arguments**

In [ ]:
args = TrainingArguments(
    output_dir = './results',          # where to save checkpoints/outputs
    eval_strategy = "epoch",           # run eval at the end of each epoch
    save_strategy = "epoch",           # save a checkpoint every epoch
    learning_rate = 2e-5,              # base learning rate
    per_device_train_batch_size = 32,  # train batch size per device
    per_device_eval_batch_size = 32,   # eval batch size per device
    num_train_epochs = 5,              # total training epochs
    weight_decay = 0.02,               # L2 weight decay strength
    load_best_model_at_end = True,     # reload best checkpoint (by metric)
    logging_dir ='./logs',             # TensorBoard log directory
    logging_steps = 10,                # log every N training steps
    report_to="tensorboard"            # send logs to TensorBoard
)

## **6. Fine-Tune Model**

In [ ]:
trainer = Trainer(
    model = model,                # Model to be trained
    args = args,                  # Training settings and hyperparameters
    train_dataset = train_ts,     # Training data
    eval_dataset = val_ts,        # Validation data
    tokenizer = tokenizer,        # Tokenizer used for preprocessing
)


In [ ]:
trainer.train()

## **7. Evaluate**

In [ ]:
want_keys = ["input_ids", "attention_mask"]              # keys the model expects (add "token_type_ids" for BERT if present)
batch = {k: torch.tensor(val_ts[k]) for k in want_keys}  # convert dataset columns (lists) into torch tensors
batch = {k: v.to(device) for k, v in batch.items()}      # move tensors to the target device (CPU/GPU)

with torch.no_grad():                                    # disable gradient tracking for inference
    outputs = model(**batch)                             # forward pass; batch must be a dict of tensors
logits = outputs.logits                                  # model scores, shape: [N, num_labels]
pred  = logits.argmax(dim=-1)                            # predicted class ids (argmax over label dimension)

In [ ]:
pred = pred.cpu().numpy()
print(confusion_matrix(val_ts['label'], pred))
print(classification_report(val_ts['label'], pred, target_names = label_list))

In [ ]:
val_org['pred'] = pred
val_org.head()

In [ ]:
# Get unique aspect categories in the validation set
asp = val_org['aspect'].unique()
asp

In [ ]:
results = []  # List to store metrics for each aspect

for aspect in asp:
    sub = val_org[val_org['aspect'] == aspect]  # Filter rows for the current aspect
    precision = precision_score(sub['label'], sub['pred'], zero_division=0)  # Aspect-wise precision
    recall = recall_score(sub['label'], sub['pred'], zero_division=0)        # Aspect-wise recall
    f1 = f1_score(sub['label'], sub['pred'], zero_division=0)                # Aspect-wise F1-score
    results.append({'aspect': aspect, 'precision': precision, 'recall': recall, 'f1': f1})

# Create a DataFrame of metrics and sort by aspect name
df_scores = pd.DataFrame(results).sort_values('aspect')

# Convert to long format for easier plotting with seaborn
df_melted = df_scores.melt(id_vars='aspect', var_name='metric', value_name='score')

In [ ]:
plt.figure(figsize=(10, 5))
# Draw a bar plot of scores for each aspect and metric
sns.barplot(data=df_melted, x='aspect', y='score', hue='metric')

plt.xticks(rotation=45)  # Rotate x-axis labels for readability
plt.ylim(0, 1.3)        # Set y-axis range to show bars clearly
plt.legend()
plt.grid()
plt.show()
